# Comparação de Modelos — BlueFlags

Este notebook compara quatro classificadores treinados no mesmo dataset pré-processado,  
usando as mesmas métricas e protocolo de avaliação dos notebooks anteriores.

### Modelos avaliados
| # | Modelo | Observação |
|---|---|---|
| 1 | **Regressão Logística** (ajustada) | Baseline — melhor config do notebook 04 |
| 2 | **Random Forest** | Ensemble de árvores, robusto a desbalanceamento |
| 3 | **Gradient Boosting** | Boosting sequencial, bom para classes minoritárias |
| 4 | **SVM (kernel RBF)** | Alta capacidade de separação em espaços de alta dimensão |

### Etapas
1. Importações e configuração  
2. Carregar dados processados  
3. Definir os modelos  
4. Treinar e avaliar todos os modelos  
5. Curvas ROC e Precisão-Recall  
6. Matrizes de confusão  
7. Tabela comparativa final  
8. Análise de erros — falsos negativos  
9. Resumo  

---
## 1. Importações e Configuração

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

# Modelling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
)

# Visualization
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from IPython.display import display

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.titlesize': 13, 'figure.dpi': 100})
pd.set_option('display.float_format', '{:.4f}'.format)

# -- Project root resolution -------------------------------------------------
def find_project_root(marker: str = 'requirements.txt') -> Path:
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root (missing '{marker}').")

PROJECT_ROOT  = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR   = PROJECT_ROOT / 'assets'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print(f'Project root : {PROJECT_ROOT}')
print('Imports complete.')

---
## 2. Carregar Dados Processados

Os arquivos foram gerados pelo `02_preprocessing.ipynb`.  
O conjunto de treino está balanceado pelo SMOTE (935 vs 935).  
O conjunto de teste preserva a distribuição original (234 vs 6).

In [ ]:
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

print('Data loaded successfully.')
print(f'  X_train : {X_train.shape}  |  class 0: {(y_train==0).sum()}  class 1: {(y_train==1).sum()}')
print(f'  X_test  : {X_test.shape}   |  class 0: {(y_test==0).sum()}   class 1: {(y_test==1).sum()}')
print(f'  Features: {list(X_train.columns)}')

---
## 3. Definir os Modelos

Todos os modelos usam `class_weight='balanced'` ou `scale_pos_weight` equivalente  
como camada de proteção adicional contra o desbalanceamento de classes,  
além do SMOTE já aplicado no conjunto de treino.

| Parâmetro-chave | Justificativa |
|---|---|
| `class_weight='balanced'` (LR, SVM) | Penaliza erros na classe minoritária proporcionalmente |
| `class_weight='balanced_subsample'` (RF) | Recalcula o peso a cada árvore — mais robusto |
| `n_estimators=300` (RF, GB) | Suficiente para convergência em dataset de ~2k amostras |
| `probability=True` (SVM) | Necessário para gerar probabilidades e calcular ROC-AUC |

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(
        C=0.1,
        penalty='l2',
        solver='lbfgs',
        max_iter=2000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=RANDOM_STATE,
    ),
    'SVM (RBF)': SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=RANDOM_STATE,
    ),
}

print(f'{len(MODELS)} models defined:')
for name in MODELS:
    print(f'  - {name}')

---
## 4. Treinar e Avaliar Todos os Modelos

Para cada modelo:
1. Treina no conjunto de treino balanceado pelo SMOTE
2. Gera probabilidades no conjunto de teste original
3. Busca o threshold que maximiza o F1-score
4. Registra ROC-AUC, Precisão Média e melhor F1

In [ ]:
THRESHOLDS = np.arange(0.05, 0.95, 0.01)

results = {}  # name -> dict of metrics + artifacts

for name, model in MODELS.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    # Best F1 threshold
    f1_scores = [
        f1_score(y_test, (y_prob >= t).astype(int), zero_division=0)
        for t in THRESHOLDS
    ]
    best_idx       = int(np.argmax(f1_scores))
    best_threshold = THRESHOLDS[best_idx]
    best_f1        = f1_scores[best_idx]
    y_pred         = (y_prob >= best_threshold).astype(int)

    roc_auc   = roc_auc_score(y_test, y_prob)
    avg_prec  = average_precision_score(y_test, y_prob)
    recall_1  = y_pred[y_test == 1].sum() / (y_test == 1).sum()

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    prec, rec, _ = precision_recall_curve(y_test, y_prob)

    results[name] = {
        'model'         : model,
        'y_prob'        : y_prob,
        'y_pred'        : y_pred,
        'roc_auc'       : roc_auc,
        'avg_precision' : avg_prec,
        'best_f1'       : best_f1,
        'best_threshold': best_threshold,
        'recall_class1' : recall_1,
        'fpr'           : fpr,
        'tpr'           : tpr,
        'precision_curve': prec,
        'recall_curve'  : rec,
    }
    print(f'done  |  ROC-AUC: {roc_auc:.4f}  AP: {avg_prec:.4f}  F1: {best_f1:.4f}  (thr={best_threshold:.2f})')

print('\nAll models trained.')

---
## 5. Curvas ROC e Precisão-Recall

- **ROC**: mede separabilidade geral — útil para comparar modelos independentemente do threshold.
- **Precisão-Recall**: mais informativa para datasets desbalanceados. A linha tracejada representa um classificador aleatório (AP = prevalência da classe = 2,5%).

In [ ]:
PALETTE = sns.color_palette('Set2', n_colors=len(MODELS))
baseline_ap = y_test.mean()  # random classifier baseline

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for (name, r), color in zip(results.items(), PALETTE):
    # ROC
    axes[0].plot(
        r['fpr'], r['tpr'], color=color, lw=2,
        label=f"{name} (AUC = {r['roc_auc']:.3f})"
    )
    # Precision-Recall
    axes[1].plot(
        r['recall_curve'], r['precision_curve'], color=color, lw=2,
        label=f"{name} (AP = {r['avg_precision']:.3f})"
    )

# Reference lines
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curvas ROC — Todos os Modelos')
axes[0].legend(loc='lower right', fontsize=9)

axes[1].axhline(baseline_ap, color='k', linestyle='--', lw=1,
                label=f'Random (AP = {baseline_ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curvas Precisão-Recall — Todos os Modelos')
axes[1].legend(loc='upper right', fontsize=9)

plt.suptitle('Comparação de Curvas de Desempenho', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_roc_pr_comparison.png', bbox_inches='tight')
plt.show()

---
## 6. Matrizes de Confusão

Cada matriz usa o threshold otimizado para F1-score.  
Com apenas 6 casos positivos no teste, cada célula tem peso significativo.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, r), color in zip(axes, results.items(), PALETTE):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d',
        cmap=sns.light_palette(color, as_cmap=True),
        ax=ax,
        xticklabels=['Sem\nDepressão', 'Depressão'],
        yticklabels=['Sem\nDepressão', 'Depressão'],
        cbar=False,
        linewidths=0.5,
        linecolor='white',
    )
    tp = cm[1, 1]
    fn = cm[1, 0]
    ax.set_title(
        f'{name}\n'
        f'F1={r["best_f1"]:.3f}  thr={r["best_threshold"]:.2f}\n'
        f'Detectados: {tp}/6 casos',
        fontsize=10,
    )
    ax.set_ylabel('Real')
    ax.set_xlabel('Previsto')

plt.suptitle('Matrizes de Confusão — Threshold Otimizado para F1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '14_confusion_matrices_comparison.png', bbox_inches='tight')
plt.show()

---
## 7. Tabela Comparativa Final

Resumo das métricas de todos os modelos, ordenados por ROC-AUC (descendente).  
O modelo com maior **Recall (Depressão)** é o preferível para triagem clínica —  
falsos negativos (casos não detectados) são mais críticos que falsos positivos.

In [ ]:
rows = []
for name, r in results.items():
    cm = confusion_matrix(y_test, r['y_pred'])
    tp, fn, fp, tn = cm[1,1], cm[1,0], cm[0,1], cm[0,0]
    rows.append({
        'Modelo'               : name,
        'ROC-AUC'              : r['roc_auc'],
        'Precisão Média (AP)'  : r['avg_precision'],
        'Melhor F1'            : r['best_f1'],
        'Threshold'            : r['best_threshold'],
        'Recall (Depressão)'   : r['recall_class1'],
        'TP (Depressão)'       : tp,
        'FN (Não detectados)'  : fn,
        'FP (Falsos alarmes)'  : fp,
    })

comparison_df = (
    pd.DataFrame(rows)
    .sort_values('ROC-AUC', ascending=False)
    .reset_index(drop=True)
)

print('Comparação de Modelos (ordenado por ROC-AUC):')
display(
    comparison_df.style
    .highlight_max(subset=['ROC-AUC', 'Precisão Média (AP)', 'Melhor F1', 'Recall (Depressão)', 'TP (Depressão)'],
                   color='#c6efce')
    .highlight_min(subset=['FN (Não detectados)', 'FP (Falsos alarmes)'],
                   color='#c6efce')
    .format({
        'ROC-AUC': '{:.4f}',
        'Precisão Média (AP)': '{:.4f}',
        'Melhor F1': '{:.4f}',
        'Threshold': '{:.2f}',
        'Recall (Depressão)': '{:.4f}',
    })
)

### Visualização das métricas principais

In [ ]:
metrics_to_plot = ['ROC-AUC', 'Precisão Média (AP)', 'Melhor F1', 'Recall (Depressão)']
plot_df = comparison_df[['Modelo'] + metrics_to_plot].set_index('Modelo')

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)

for ax, metric, color in zip(axes, metrics_to_plot, PALETTE):
    vals = plot_df[metric].sort_values(ascending=False)
    bars = ax.barh(vals.index, vals.values,
                   color=[color if v == vals.max() else '#d3d3d3' for v in vals.values],
                   edgecolor='white', height=0.6)
    ax.set_xlim(max(0, vals.min() - 0.05), min(1.0, vals.max() + 0.05))
    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.set_xlabel('Score')
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.invert_yaxis()

plt.suptitle('Desempenho por Métrica — Melhor modelo destacado em cor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '15_metrics_bar_comparison.png', bbox_inches='tight')
plt.show()

---
## 8. Análise de Erros — Falsos Negativos

Com apenas 6 casos reais de depressão no conjunto de teste, cada falso negativo  
(caso não detectado) representa uma falha crítica para o objetivo clínico do modelo.  

Esta seção identifica quais casos de depressão **nenhum modelo conseguiu detectar**  
e quais foram detectados consistentemente.

In [ ]:
# Positive cases in the test set
positive_idx = y_test[y_test == 1].index

# Build a prediction agreement table for positive cases only
pred_matrix = pd.DataFrame(
    {name: results[name]['y_pred'][positive_idx] for name in results},
    index=positive_idx,
)
prob_matrix = pd.DataFrame(
    {name: results[name]['y_prob'][positive_idx] for name in results},
    index=positive_idx,
)

pred_matrix['Detectado por (N modelos)'] = pred_matrix.sum(axis=1)
pred_matrix['Prob. média'] = prob_matrix.mean(axis=1).round(4)
pred_matrix['Prob. mín.']  = prob_matrix.min(axis=1).round(4)

print('Casos positivos no teste (depression_label = 1):')
print('1 = detectado como depressão  |  0 = não detectado (falso negativo)\n')
display(pred_matrix)

# Summary
n_total = len(positive_idx)
all_detected = (pred_matrix['Detectado por (N modelos)'] == len(MODELS)).sum()
none_detected = (pred_matrix['Detectado por (N modelos)'] == 0).sum()

print(f'\nTotal de casos positivos no teste : {n_total}')
print(f'Detectados por TODOS os modelos   : {all_detected}')
print(f'Não detectados por NENHUM modelo  : {none_detected}')

In [ ]:
# Feature profile of positive cases: detected vs. missed (aggregated)
# Uses the model with the best Recall as reference
best_recall_model = comparison_df.sort_values('Recall (Depressão)', ascending=False).iloc[0]['Modelo']
y_pred_best = results[best_recall_model]['y_pred']

pos_mask    = y_test == 1
detected    = X_test.loc[pos_mask & (pd.Series(y_pred_best, index=y_test.index) == 1)]
missed      = X_test.loc[pos_mask & (pd.Series(y_pred_best, index=y_test.index) == 0)]

numeric_cols = ['stress_level', 'anxiety_level', 'sleep_hours',
                'daily_social_media_hours', 'screen_time_before_sleep']

if len(missed) > 0 and len(detected) > 0:
    profile = pd.DataFrame({
        f'Detectados ({len(detected)})' : detected[numeric_cols].mean(),
        f'Não detectados ({len(missed)})': missed[numeric_cols].mean(),
    })
    print(f'Perfil médio das features — casos positivos ({best_recall_model}):')
    display(profile)
elif len(missed) == 0:
    print(f'✅  {best_recall_model} detectou todos os {len(detected)} casos de depressão no teste.')
else:
    print(f'❌  {best_recall_model} não detectou nenhum caso de depressão.')

---
## 9. Resumo

In [ ]:
best_roc    = comparison_df.sort_values('ROC-AUC', ascending=False).iloc[0]
best_f1_row = comparison_df.sort_values('Melhor F1', ascending=False).iloc[0]
best_rec    = comparison_df.sort_values('Recall (Depressão)', ascending=False).iloc[0]

print('=' * 65)
print('          RESUMO — COMPARAÇÃO DE MODELOS')
print('=' * 65)
print(f'  Dataset de treino  : {len(X_train):,} amostras (SMOTE balanceado)')
print(f'  Dataset de teste   : {len(X_test):,} amostras | {int((y_test==1).sum())} casos de depressão')
print()
print('  Rankings:')
print(f'    Maior ROC-AUC           : {best_roc["Modelo"]:<25} {best_roc["ROC-AUC"]:.4f}')
print(f'    Maior F1-Score          : {best_f1_row["Modelo"]:<25} {best_f1_row["Melhor F1"]:.4f}')
print(f'    Maior Recall Depressão  : {best_rec["Modelo"]:<25} {best_rec["Recall (Depressão)"]:.4f}')
print()
print('  Resultados por modelo:')
for _, row in comparison_df.iterrows():
    print(f'    {row["Modelo"]:<26}  '
          f'AUC={row["ROC-AUC"]:.4f}  '
          f'F1={row["Melhor F1"]:.4f}  '
          f'Recall={row["Recall (Depressão)"]:.2f}  '
          f'TP={int(row["TP (Depressão)"])}/6')
print()
print('  Recomendação para uso clínico:')
print(f'    → Priorizar {best_rec["Modelo"]} — máximo recall, menor risco de')
print(f'      não detectar casos reais de depressão.')
print('=' * 65)